# unbind-tuple-unpack — ex2: split packed QKV stack into named Q, K, V tensors

> Procedural drill from [Delta Drills](https://delta-drills.vercel.app).
> Atom: `unbind-tuple-unpack`. Running the final beacon cell reports progress against the `PyTorch: Unbind tuple-unpack` subtopic.

**Why this is a Colab exercise.** This standalone exercises material the Delta Drills flashcards can't deliver on their own — interactive tensor execution, visualization, or multi-step debugging. Read the prompt, fill in the function body, run the test cell, then run the beacon at the bottom.

## Setup

In [ ]:
import numpy as np
import torch as t
from torch import Tensor
import einops
from einops import rearrange, reduce, repeat

t.manual_seed(0)
np.random.seed(0)

## Connect to Delta Drills

Paste your Delta Drills auth token below so this drill can report progress on the `PyTorch: Unbind tuple-unpack` subtopic. Copy it from your Delta Drills account page.

This standalone exercises the atom **`unbind-tuple-unpack`** (exercise 2). Completion fires the beacon at the bottom.

In [ ]:
# === Delta Drills auth ===
DD_TOKEN = ""  # paste your token here, then run this cell
DD_ATOM_ID = "unbind-tuple-unpack"
DD_SUBTOPIC = "PyTorch: Unbind tuple-unpack"
DD_BACKEND_URL = "https://delta-drills-backend.fly.dev"

_dd_passed = set()

## unbind tuple-unpack — quick refresher

`x.unbind(dim=k)` returns a **Python tuple** of `x.shape[k]` view-tensors with axis `k` removed. Tuple-unpacking the result gives you named slices without any indexing arithmetic:

```python
q, k, v = qkv_stack.unbind(dim=0)
```

**This drill (ex2) vs ex1.** ex1 destructured rays `(B,2,3)` into `ox/oy/oz/dx/dy/dz` (a two-level decomposition on a small leading axis). ex2 destructures the canonical QKV-stack-leading-axis pattern from a packed attention projection — same single `unbind` op, different semantic axis (a length-3 *triple-of-tensors* leading axis, not a geometric `xyz` axis).

### Exercise 2 — split packed QKV stack into named Q, K, V tensors

> ```yaml
> Difficulty: 🔴🔴🔴⚪⚪
> Bloom level: Apply
> LO: Apply `unbind(dim=0)` to a packed `(3, B, H, S, D)` QKV-stack tensor from a fused attention projection, producing three named `(B, H, S, D)` tensors `q, k, v` ready for per-tensor downstream ops.
> Keywords: attention, qkv, unbind-dim-0, named-destructure
> ```

**KCs targeted:** `unbind-returns-python-tuple`, `unbind-leading-axis-destructure`

Implement `ex2_split_qkv(qkv)`.

A fused attention projection produces a single tensor `qkv` of shape `(3, B, H, S, D)` — the leading length-3 axis stacks Q, K, V (in that order). Split it into three named tensors with `unbind`.

**Rules.**
1. Use `qkv.unbind(dim=0)` and tuple-unpack the result.
2. Return a dict `{'q': ..., 'k': ..., 'v': ...}` so the caller can name-address each tensor.
3. Each value must have shape `(B, H, S, D)`. No reshape / index arithmetic — destructure only.
4. Verify the views still share storage with `qkv` (they're views, not copies). The test asserts this via `data_ptr()`.

Inputs:
- `qkv`: `(3, B, H, S, D)` float tensor.

Output: dict with keys `'q'`, `'k'`, `'v'`, each a `(B, H, S, D)` view.

In [ ]:
def ex2_split_qkv(qkv: Tensor) -> dict:
    q, k, v = qkv.unbind(dim=0)
    return {'q': q, 'k': k, 'v': v}


<details><summary>Solution</summary>

```python
def ex2_split_qkv(qkv: Tensor) -> dict:
    q, k, v = qkv.unbind(dim=0)
    return {'q': q, 'k': k, 'v': v}
```

**Why `unbind(dim=0)` and not `qkv[0], qkv[1], qkv[2]`.** Both work, but `unbind` makes the destructure intent explicit at the call-site ("peel the leading axis into a tuple") and composes cleanly with Python tuple-unpack. The indexed form looks like three independent operations even though they're one decomposition.

**Why this is the canonical QKV pattern.** Fused attention projections (e.g. nanoGPT, ViT, transformer encoder blocks) produce a single `(3*D, B, ...)` projection then `view(3, D, B, ...)` and unbind. The unbind step is the natural "untie the three tensors" boundary.

**Difference from ex1.** ex1 was a two-level unbind on a `(B,2,3)` ray tensor (inner-axis-and-then-last-axis decomposition over a geometric `xyz` axis). ex2 is a single-level unbind on the LEADING length-3 axis of a packed-projection tensor — same op, different semantic axis position.
</details>

## Report completion

Run the cell below to send your progress to Delta Drills. The beacon fires only if the test cell above passed.

In [ ]:
# === Delta Drills completion beacon ===
import urllib.request as _dd_req, json as _dd_json

_DD_REQUIRED = {'ex2'}

def report_completion():
    missing = _DD_REQUIRED - _dd_passed
    if missing:
        print(f"[Delta Drills] {sorted(missing)} not yet passing — fix the cell above, then re-run this one.")
        return
    if not DD_TOKEN:
        print('[Delta Drills] DD_TOKEN is empty — completion not reported.')
        return
    body = _dd_json.dumps({
        'exercise_title': f'procedural-drill:{DD_ATOM_ID}:ex2',
        'subtopics': [DD_SUBTOPIC],
        'feedback': 'somewhat',
        'correct': True,
    }).encode('utf-8')
    req = _dd_req.Request(
        f'{DD_BACKEND_URL}/api/practice/arena-rating',
        data=body,
        headers={
            'Content-Type': 'application/json',
            'Authorization': f'Bearer {DD_TOKEN}',
        },
        method='POST',
    )
    try:
        with _dd_req.urlopen(req, timeout=5) as r:
            resp = _dd_json.loads(r.read())
        print(f'[Delta Drills] reported {DD_ATOM_ID} (subtopic={DD_SUBTOPIC!r})')
        print(f'[Delta Drills] EWMA updated: {resp}')
    except Exception as e:
        print(f'[Delta Drills] beacon failed: {e}')

report_completion()